# Recommendation model

In [48]:
import os
from dotenv import load_dotenv
load_dotenv()
from huggingface_hub import login,InferenceClient
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_huggingface import HuggingFaceEmbeddings,HuggingFaceEndpoint
from langchain_chroma import Chroma
import json

EMBEDDING_MODEL='sentence-transformers/all-MiniLM-L6-v2'
LLM_MODEL_NAME = "Qwen/Qwen2.5-72B-Instruct"
# LLM_MODEL_NAME = "DeepSeek-R1-Distill-Qwen-32B"
HF_TOKEN = os.getenv("HF_TOKEN")

#### Import embeddings and create vectorstore from database

In [52]:
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
vectorstore = Chroma(
    persist_directory="C:\\Users\\abhik\\Desktop\\PROJECTS_FOR_CV\\Recommendation_engine_LLM\\data\\chroma_db",   # folder containing chroma.sqlite3
    embedding_function=embeddings
)
# check vectorstore
test_docs = vectorstore.similarity_search("python", k=5)
print(f"Total documents in Chroma: {vectorstore._collection.count()}")
print(f"Test search returned {len(test_docs)} documents.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Total documents in Chroma: 199120
Test search returned 5 documents.


### Main logic

1 extract_filters_with_llm: Uses the LLM to understand text intent.

2 search_courses: Searches ChromaDB and applies Python metadata filters.

3 resolve_filters_and_search: Connects them together.

#### All default search parameters

In [32]:
DEFAULT_SEARCH_FILTERS = {
    "query": "courses","course_type": "All","max_price": 1000.0,"language": "All","subcategory": "All",
    "duration": "All","min_rating": 0.0,"sort_by": "Most Relevant","top_k": 5
}

#### Filter Search by LLM

In [43]:
SYSTEM_PROMPT = """
You are an intent extraction engine for a course search tool.
Extract explicit search criteria from the user's query into JSON.

Valid values:
- "query": Cleaned semantic topic string (remove keywords like free, top, arabic, etc.).
- "course_type": "Free", "Paid", or "All"
- "max_price": float (default 1000.0)
- "language": "English", "Arabic", "Chinese", etc., or "All"
- "duration": "Short", "Medium", "Long", or "All"
- "min_rating": float (0.0 to 5.0, default 0.0)
- "sort_by": "Latest First" or "Most Relevant"
- "top_k": integer (default 5)

Output ONLY raw JSON. No explanation, no markdown tags.
"""
def extract_filters_with_llm(user_query):
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL_NAME,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_query}
            ],
            max_tokens=250,
            temperature=0.0
        )
        return json.loads(response.choices[0].message.content.strip())
    except Exception:
        return {}

#### Filter Search by UI/keywords

In [ ]:
def search_courses(search_filters):

    raw_results = vectorstore.similarity_search(query=search_filters["query"], k=search_filters["top_k"] * 5) # Fetch extra candidates for safe filtering
    filtered_courses = []
    
    for doc in raw_results:
        meta = doc.metadata
        
        # Filter: Free vs Paid
        if search_filters["course_type"] == "Free" and meta["is_paid"] != "Free":
            continue
        if search_filters["course_type"] == "Paid" and meta["is_paid"] == "Free":
            continue

        # Filter: Max Price
        if search_filters["course_type"] == "Paid" and float(meta["price"]) > float(search_filters["max_price"]):
            continue

        # Filter: Language
        if search_filters["language"] != "All" and meta["language"] != search_filters["language"]:
            continue

        # Filter: Subcategory
        if search_filters["subcategory"] != "All" and meta["subcategory"] != search_filters["subcategory"]:
            continue

        # Filter: Duration
        if search_filters["duration"] != "All" and meta["content_length_minutes"] != search_filters["duration"]:
            continue

        # Filter: Minimum Rating
        if float(meta["avg_rating"]) < float(search_filters["min_rating"]):
            continue

        # Map clean metadata
        filtered_courses.append({
            "title": meta["title"],
            "is_paid": meta["is_paid"],
            "price": "Free" if meta["is_paid"] == "Free" else f"${meta['price']}",
            "num_subscribers": meta["num_subscribers"],
            "avg_rating": round(float(meta["avg_rating"]), 2),
            "published_time": meta["published_time"],
            "subcategory": meta["subcategory"],
            "language": meta["language"],
            "course_url": meta["course_url"],
            "content_length_minutes": meta["content_length_minutes"]
        })

    
    if search_filters["sort_by"] == "Latest First":
        filtered_courses.sort(key=lambda x: x["published_time"], reverse=True)

    return filtered_courses[:search_filters["top_k"]] # Return final top_k slice
    # filtered_courses is a LIST of dictionaries e.g. [Dictionary for Course 1, Dictionary for Course 2 etc]

#### Final function - calling both Filter search by LLM and filter search by UI/keywords

In [35]:
def get_courses(user_prompt="", ui_filters=None):
    search_filters = DEFAULT_SEARCH_FILTERS.copy()

    # 1. Update from UI choices
    if ui_filters:
        for key, value in ui_filters.items():
            if value is not None:
                search_filters[key] = value

    # 2. Update from LLM extraction if user entered a query string
    if user_prompt and user_prompt.strip():
        llm_extracted = extract_filters_with_llm(user_prompt)
        for key, value in llm_extracted.items():
            if value:
                search_filters[key] = value

    # 3. Execute search
    return search_courses(search_filters)

#### LLM Intro & Outro Generation

In [44]:
client = InferenceClient(api_key=os.environ["HF_TOKEN"])
def generate_llm_response(prompt: str, max_tokens: int = 200) -> str:
    """Sends prompt to Hugging Face Inference API."""
    try:
        completion = client.chat.completions.create(
            model=LLM_MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=0.7
        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        print(f"LLM API Error: {e}")
        return ""

def generate_intro(user_query: str, num_courses: int) -> str:
    prompt = f"Generate a friendly, brief conversational introduction for course recommendations. User Query: '{user_query}', Number of courses found: {num_courses}."
    response = generate_llm_response(prompt)
    return response if response else f"Here are the top {num_courses} courses matching your search:"

def generate_outro(user_query: str) -> str:
    prompt = f"Generate a brief closing sentence for a course recommendation list based on user query: '{user_query}'."
    response = generate_llm_response(prompt)
    return response if response else "Hope these recommendations help you find the right course!"

# Evaluation

In [50]:
import os

# 1. Test Environment Variables
print("--- 1. Testing Setup ---")
print(f"HF Token loaded: {bool(os.getenv('HF_TOKEN'))}")
print(f"Embedding Model: {EMBEDDING_MODEL}")
print(f"LLM Model: {LLM_MODEL_NAME}")

# Verify ChromaDB connection count
doc_count = vectorstore._collection.count()
print(f"Total documents in ChromaDB: {doc_count}")
assert doc_count > 0, "ChromaDB is empty! Check database path."

# 2. Test LLM Connection (Intro/Outro)
print("\n--- 2. Testing LLM Direct Generation ---")
test_intro = generate_intro(user_query="python for beginners", num_courses=3)
print(f"Generated Intro:\n-> {test_intro}\n")

test_outro = generate_outro(user_query="python for beginners")
print(f"Generated Outro:\n-> {test_outro}\n")

# 3. Test Intent Extraction
print("--- 3. Testing LLM Intent Extraction ---")
test_query = "share top 3 free arabic python courses"
extracted = extract_filters_with_llm(test_query)
print(f"Input Query: '{test_query}'")
print(f"Extracted JSON from LLM: {extracted}\n")

# 4. Test Master Function (get_courses)
print("--- 4. Testing End-to-End Search Pipeline (get_courses) ---")

# Condition A: Text Query Only
print("Scenario A: Query string provided")
results_a = get_courses(user_prompt=test_query)
print(f"Found {len(results_a)} course(s):")
for idx, c in enumerate(results_a, 1):
    print(f"  {idx}. {c['title']} | Language: {c['language']} | Price: {c['price']}")

# Condition B: UI Filters Only (No text query)
print("\nScenario B: UI filters provided without text query")
sample_ui_filters = {
    "course_type": "Free",
    "language": "English",
    "top_k": 2
}
results_b = get_courses(user_prompt="", ui_filters=sample_ui_filters)
print(f"Found {len(results_b)} course(s):")
for idx, c in enumerate(results_b, 1):
    print(f"  {idx}. {c['title']} | Language: {c['language']} | Price: {c['price']}")

print("\n All backend tests completed successfully!")

--- 1. Testing Setup ---
HF Token loaded: True
Embedding Model: sentence-transformers/all-MiniLM-L6-v2
LLM Model: Qwen/Qwen2.5-72B-Instruct
Total documents in ChromaDB: 199120

--- 2. Testing LLM Direct Generation ---
Generated Intro:
-> Hi there! I see you're interested in learning Python. That's a great choice! I found 3 beginner-friendly courses that might suit you perfectly. Would you like to hear more about them?

Generated Outro:
-> Whether you're just starting out or looking to solidify your foundational skills, these Python for beginners courses are your perfect launchpad into the world of programming.

--- 3. Testing LLM Intent Extraction ---
Input Query: 'share top 3 free arabic python courses'
Extracted JSON from LLM: {'query': 'python', 'course_type': 'Free', 'max_price': 0.0, 'language': 'Arabic', 'duration': 'All', 'min_rating': 0.0, 'sort_by': 'Most Relevant', 'top_k': 3}

--- 4. Testing End-to-End Search Pipeline (get_courses) ---
Scenario A: Query string provided
Found

In [49]:
print("Testing Direct Generation:")
print(generate_intro("python for beginners", 3))

print("\nTesting Intent Extraction:")
print(extract_filters_with_llm("share top 3 free arabic python courses"))

Testing Direct Generation:
Hi there! Looks like you're interested in getting started with Python. I've found 3 great beginner-friendly courses that might suit you perfectly. Would you like a quick rundown of each one?

Testing Intent Extraction:
{'query': 'python', 'course_type': 'Free', 'max_price': 0.0, 'language': 'Arabic', 'duration': 'All', 'min_rating': 0.0, 'sort_by': 'Most Relevant', 'top_k': 3}


In [51]:
# Test 1: Search for Free English Python courses (Should find results!)
print("--- Test 1: Free English Python Courses ---")
test_query_1 = "share top 3 free english python courses"
results_1 = get_courses(user_prompt=test_query_1)

print(f"Found {len(results_1)} course(s):")
for idx, c in enumerate(results_1, 1):
    print(f"  {idx}. {c['title']} | Language: {c['language']} | Price: {c['price']}")

# Test 2: Search for Paid Python courses
print("\n--- Test 2: Top 3 Python Courses ---")
test_query_2 = "top 3 python courses"
results_2 = get_courses(user_prompt=test_query_2)

print(f"Found {len(results_2)} course(s):")
for idx, c in enumerate(results_2, 1):
    print(f"  {idx}. {c['title']} | Language: {c['language']} | Price: {c['price']}")

--- Test 1: Free English Python Courses ---
Found 3 course(s):
  1. introduction to programming with python | Language: English | Price: Free
  2. introduction to python programming | Language: English | Price: Free
  3. learn python basics | Language: English | Price: Free

--- Test 2: Top 3 Python Courses ---
Found 3 course(s):
  1. introduction to programming with python | Language: English | Price: Free
  2. python 3 programación orientada a objetos | Language: Spanish | Price: $49.99
  3. python technical interview questions the ultimate course | Language: English | Price: $199.99


In [ ]:
# answer_query("share best 3 arabic courses")
# answer_query("share top 3 automobile courses")
# answer_query("share top 10 chinese courses")
# answer_query("share top 10 chinese courses in chinese language")
# answer_query("share widely recognise 6 chinese courses")
# answer_query('suggest data science courses')
# answer_query('Can you suggest affordable, popular 7 data science courses?')
# answer_query('Can you suggest affordable, popular 5 data science courses?')
# answer_query("Please share the top ten widely recognized criminal law courses")
# answer_query("Please share the top 10 widely recognized criminal law courses")
# answer_query("hello give me best courses")
# answer_query("Hi there, what are the latest top 10 english courses available for civil engineering and java?")
# answer_query("Hi there, what are the cheap 15 english courses available for civil engineering and java?")
# answer_query("let me know cheap courses available for photography?")
# answer_query("will you give me top 6 highly rated gaming courses")
# answer_query("Please share the top 8 widely recognized free chinese courses")
# answer_query("Please share the top 3 medium courses")
# answer_query("Please share the most popular 2 medium arabic courses")
# answer_query("Provde marathi courses")
# answer_query("let me know lengthy hindi courses")
# answer_query("I request you to give short length spanish courses")

In [58]:
import pandas as pd

# Load the dataset directly
df = pd.read_excel("C:\\Users\\abhik\\Desktop\\PROJECTS_FOR_CV\\Recommendation_engine_LLM\\data\\udemy_data_cleaned.xlsx")

# Extract unique values directly from DataFrame columns
UNIQUE_LANGUAGES = ["All"] + sorted(df["language"].dropna().unique().tolist())
UNIQUE_SUBCATEGORIES = ["All"] + sorted(df["subcategory"].dropna().unique().tolist())

UNIQUE_LANGUAGES

['All',
 'Afrikaans',
 'Albanian',
 'Arabic',
 'Armenian',
 'Azeri',
 'Bengali',
 'Bulgarian',
 'Burmese',
 'Catalan',
 'Chinese',
 'Croatian',
 'Czech',
 'Danish',
 'Dutch',
 'English',
 'Estonian',
 'Filipino',
 'Finnish',
 'French',
 'Georgian',
 'German',
 'Greek',
 'Gujarati',
 'Haitian',
 'Hebrew',
 'Hindi',
 'Hungarian',
 'Indonesian',
 'Irish',
 'Italian',
 'Japanese',
 'Kannada',
 'Kazakh',
 'Khmer',
 'Korean',
 'Kurdish',
 'Latvian',
 'Lithuanian',
 'Macedonian',
 'Malay',
 'Malayalam',
 'Marathi',
 'Mongolian',
 'Nepali',
 'Norwegian',
 'Pashto',
 'Persian',
 'Polish',
 'Portuguese',
 'Punjabi',
 'Quechua',
 'Romanian',
 'Russian',
 'Serbian',
 'Slovak',
 'Slovenian',
 'Somali',
 'Spanish',
 'Swahili',
 'Swedish',
 'Tamil',
 'Telugu',
 'Thai',
 'Turkish',
 'Ukrainian',
 'Urdu',
 'Uzbek',
 'Vietnamese']

In [ ]:
UNIQUE_SUBCATEGORIES

In [63]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 199120 entries, 0 to 199119
Data columns (total 11 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   title                   199120 non-null  str           
 1   is_paid                 199120 non-null  str           
 2   price                   199120 non-null  float64       
 3   num_subscribers         199120 non-null  int64         
 4   avg_rating              199120 non-null  float64       
 5   published_time          199120 non-null  datetime64[us]
 6   subcategory             199120 non-null  str           
 7   language                199120 non-null  str           
 8   course_url              199120 non-null  str           
 9   content_length_minutes  199120 non-null  str           
 10  processed_text          199120 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(7)
memory usage: 16.7 MB


In [65]:
df['is_paid'].value_counts()

is_paid
Paid    177640
Free     21480
Name: count, dtype: int64

In [66]:
sample = vectorstore._collection.get(limit=1, include=["metadatas"])
print(sample["metadatas"][0])

{'avg_rating': 3.75, 'course_url': '/course/vegan-vegetarian-cooking-school/', 'price': 24.99, 'is_paid': 'Paid', 'content_length_minutes': 'Long', 'language': 'English', 'subcategory': 'Food & Beverage', 'published_time': 2010, 'title': 'online vegan vegetarian cooking school', 'num_subscribers': 2231}


In [68]:
df['price'].describe()

count    199120.000000
mean         82.775516
std         116.949366
min           0.000000
25%          19.990000
50%          39.990000
75%          99.990000
max         999.990000
Name: price, dtype: float64